In [ ]:
!pip install PyPDF2

* READ ME *

This code counts the number of instances of a concrete material in an article

1) extract_text_from_pdf - gets the text from PDF.

2) count_materials_in_text - counts which material is present in a concrete article.

3) process_folders - main function. it starts to process each folder and each article. then it calls "count_materials_in_text" function to count which material is present in each article. then it gets the article which have the material.

4) then it processes to the result and provides the output.

In [1]:
import os
import PyPDF2
from collections import defaultdict

In [ ]:
materials = ['AlSi10Mg', 'SS316L', 'Ti6Al4V']

In [ ]:
def extract_text_from_pdf(pdf_path):
    text = ""  
    file = open(pdf_path, 'rb') 
    
    try: 
        reader = PyPDF2.PdfReader(file) 
        for page in reader.pages: 
            page_text = page.extract_text() 
            if page_text: # if the text was successfully extracted
                text += page_text 
    finally:
        file.close()  

    return text.lower() 

In [ ]:
# Function that counts the unique materials
def count_materials_in_text(text, materials): 
    material_mentions = []  
    for material in materials: 
        if material in text: 
            material_mentions.append(material)  
    return material_mentions 

In [ ]:
# Function that processes all folders
def process_folders(base_folder, fields, materials): 
    
    # creating a dictionary. Key = field, value = object of type defaultdict(int) (material)
    # so initially each category will have its own material count with 0 initially
    field_material_instances = {field: defaultdict(int) for field in fields}  

    field_file_matches = defaultdict(list)  # Dictionary to store filenames (PDFs) where materials are found

    # going through each category
    for field in fields:
        folder_path = os.path.join(base_folder, field) 
        for filename in os.listdir(folder_path): 
            if filename.endswith('.pdf'): 
                file_path = os.path.join(folder_path, filename) 
                text = extract_text_from_pdf(file_path) 
                
                # just was doing some test to verify if it reads
                #print(f"Processing file: {file_path}")
                #print(f"Extracted text: {text}")  # First 500 symbols

                # getting the function to get the mentions of concrete material in this article
                material_mentions = count_materials_in_text(text, materials)

                # if are able to find some materials
                if material_mentions:
                    field_file_matches[field].append((filename, material_mentions)) 
                
                for material in material_mentions:
                    field_material_instances[field][material] += 1

    return field_material_instances, field_file_matches

In [ ]:
base_folder = 'datasets' 
fields = ['aerospace_new', 'automotive_new', 'biomedical_new'] 

lower_materials = [material.lower() for material in materials]

In [ ]:
results, file_matches  = process_folders(base_folder, fields, lower_materials)

In [ ]:
# Output
for field, instances in results.items(): 
    
    print(f"\nField: {field}")
    for material in lower_materials: 
        count = instances.get(material, 0) 
        print(f"  {material}: {count}")
    if not instances: 
        print("No materials found")

    print()


Field: aerospace_new
  alsi10mg: 247
  ss316l: 107
  ti6al4v: 309


Field: automotive_new
  alsi10mg: 323
  ss316l: 138
  ti6al4v: 297


Field: biomedical_new
  alsi10mg: 156
  ss316l: 64
  ti6al4v: 373

